# 💘 Valentine's Day Special: The Story of Love in Sweden 💘

## A Data-Driven Journey Through Marriage and Divorce Trends

**Happy Valentine's Day!** 🌹

What better way to celebrate the day of love than by diving into the fascinating data behind Swedish marriages?

Using the `pxstatspy` library, we'll explore over 150 years of romantic statistics from Statistics Sweden (SCB).

**Data Source:** Statistics Sweden (SCB)

### What We'll Discover:
- 📈 Historical trends from 2000-2025
- 👫 Age patterns in marriages and divorces
- 💍 How the average age at first marriage has evolved since 1871
- 🏳️‍🌈 Trends in same-sex marriages

Let's begin our romantic data adventure!

## 🛠️ Setup: Installing pxstatspy

First, let's install the library and import our tools!

In [ ]:
# Install pxstatspy if not already installed
!pip install pxstatspy -q

# Import necessary libraries
from pxstatspy import PxAPI, PxAPIConfig
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configure plot aesthetics for Valentine's Day theme
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ All libraries loaded successfully!")

## 🔌 Connect to Statistics Sweden API

Let's initialize our connection to the official Swedish statistics database!

In [ ]:
# Initialize the API client
config = PxAPIConfig(
    base_url="https://api.scb.se/ov0104/v2beta/api/v2",
    language="en"
)

client = PxAPI(config)

print("🎉 Successfully connected to Statistics Sweden API!")

Now that we're connected, we can start by searching for specific tables in the database by using a text query.

In [ ]:
# Search for tables in database 
client.find_tables(query="Newly married")


Let's investigate the table variables a little closer in order to later build our query for the data fetch API call!

In [ ]:
# Print the first values for each variable in the table
client.print_table_variables(
    table_id='TAB5829',
    max_values=1    # Specify how many values you want to see for each variable
)

In [ ]:
# View all values for TypPar (type of couple)
client.print_table_variables(
    table_id='TAB5829',
    variable_id='TypPar',   # Specify the variable id
    max_values='*'          # Use '*' to get all values
)

We can learn more about a table by fetching it's metadata! In this example we print the table label and notes.

In [ ]:
# Get metadata for selected table
metadata = client.get_table_metadata('TAB5829')
label = metadata['label']
notes = metadata['note']

print(f"📊 Table: {label}\n")
print("📋 Important Notes about this Dataset:\n")
for i, note in enumerate(notes[:2], 1):  # Show only first 2 notes
    # Clean up the note text
    clean_note = note.replace('\r', ' ').strip()
    print(f"{clean_note}\n")
print("... (and more)")

After some searching, we have found some interesting tables for our Valentine's Day analysis!

Here are the tables we are looking into today:

- **TAB6478**: Number of marriages and divorces per month by region. Year 2025
- **TAB5811**: Number of marriages and divorces per month by region. Year 2000-2024
- **TAB5833**: Number of marriages and divorces by region and type. Year 2000-2024
- **TAB5829**: Newly married, divorced and widowed by region, marital status, type of couple, age and sex. Year 2000-2024
- **TAB4400**: Average age at marriage by sex. Year 1871-2024

---

# 📈 Part 1: Two and a half Decades of Love - Trends from 2000 to 2025

Now let's dig in to the analysis and look at how marriage and divorce patterns have evolved over the past two and a half decades. Are there seasonal patterns? How popular is February (Valentine's month!) for weddings? 💐

In [ ]:
# Fetch Number of marriages and divorces per month by region. Year 2025
print("📥 Fetching 2025 marriage and divorce data...\n")

df_2025 = client.get_data_as_dataframe(
    table_id="TAB6478",
    value_codes={
        "Tid": ["*"],               # use '*' to get all available values for variable
        "Manad": ["RANGE(01,12)"],  # with RANGE() you can get all values in specified range
        "Region": ["00"],
        "ContentsCode": ["000007T7", "000007T8"]
    }
)

# Fetch Number of marriages and divorces per month by region. Year 2000-2024
print("📥 Fetching historical marriage and divorce data from 2000-2024...\n")

df_2000_2024 = client.get_data_as_dataframe(
    table_id="TAB5811",
    value_codes={
        "Tid": ["*"],
        "Manad": ["RANGE(01,12)"],
        "Region": ["00"],
        "ContentsCode": ["0000052L", "0000052M"]
    }
)

print("\n✅ Successfully downloaded data from both tables")

In [ ]:
# Prepare data from both datasets
df_hist = df_2000_2024.copy()
df_hist.columns = ['region_code', 'region', 'month', 'year', 'marriages', 'divorces']

df_current = df_2025.copy()
df_current.columns = ['region_code', 'region', 'month', 'year', 'marriages', 'divorces']

# Union both datasets
df_combined = pd.concat([df_hist, df_current], ignore_index=True)

# Create month number for sorting
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
df_combined['month_num'] = df_combined['month'].map({m: i for i, m in enumerate(month_order, 1)})
df_combined = df_combined.sort_values(['year', 'month_num'])

# Calculate average marriages and divorces per month across all years (2000-2025)
avg_by_month = df_combined.groupby('month')[['marriages', 'divorces']].mean().reset_index()
avg_by_month['month_num'] = avg_by_month['month'].map({m: i for i, m in enumerate(month_order, 1)})
avg_by_month = avg_by_month.sort_values('month_num')

# Create single plot with both marriages and divorces
fig, ax = plt.subplots(figsize=(15, 8))

# Plot average marriages and divorces
ax.plot(avg_by_month['month'], avg_by_month['marriages'], 
        marker='o', linewidth=3, label='Average Marriages', 
        markersize=10, color='#FF69B4')
ax.plot(avg_by_month['month'], avg_by_month['divorces'], 
        marker='s', linewidth=3, label='Average Divorces', 
        markersize=10, color='#4682B4', linestyle='--')

ax.set_title('Average number of Swedish Marriages vs Divorces by Month (2000-2025)', 
              fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Number of Events', fontsize=12, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Calculate seasonal patterns
print("\n🌸 Seasonal Insights:")
seasonal_marriages = avg_by_month.set_index('month')['marriages'].sort_values(ascending=False)
print(f"\n💒 Top 3 months for marriages (average 2000-2025):")
for i, (month, count) in enumerate(seasonal_marriages.head(3).items(), 1):
    print(f"  {i}. {month}: {count:,.0f} marriages")

print(f"\n💝 Is February popular for weddings? {'YES! 🎉' if seasonal_marriages['February'] > seasonal_marriages.mean() else 'Not particularly 😅'}")
print(f"   Average marriages in February: {seasonal_marriages['February']:,.0f}")
print(f"   Overall monthly average: {seasonal_marriages.mean():,.0f}")

seasonal_divorces = avg_by_month.set_index('month')['divorces'].sort_values(ascending=False)
print(f"\n💔 Top 3 months for divorces (average 2000-2025):")
for i, (month, count) in enumerate(seasonal_divorces.head(3).items(), 1):
    print(f"  {i}. {month}: {count:,.0f} divorces")

---

# 👫 Part 2: The Demographics of Love - Who's Getting Married and Divorced?

Let's explore the age patterns and types of couples in Swedish marriages and divorces from 2000 to 2024. This includes the beautiful diversity of love, including same-sex marriages since the gender neutral marriage law in 2009! 🏳️‍🌈

In [ ]:
# Fetch data for all couple types
print("📥 Fetching marriage data by couple type (2000-2024)...\\n")

df_new_two_sexes = client.get_data_as_dataframe(
    table_id="TAB5833",
    value_codes={
        "Tid": ["*"],
        "Region": ["00"],
        "TypPar": ["*"],
        "ContentsCode": ["0000053Q"]
    }
)

print("\n✅ Successfully downloaded the data")

In [ ]:
# Prepare data
df_couples = df_new_two_sexes.copy()
df_couples.columns = ['region_code', 'region', 'type of couple', 'year', 'Number']

# Convert year to numeric for proper plotting
df_couples['year'] = pd.to_numeric(df_couples['year'])

# Filter data from 2009 onwards (when same-sex marriage became legal)
df_couples_filtered = df_couples[df_couples['year'] >= 2009].copy()

# Separate same-sex and two-sex couples
same_sex_marriages = df_couples_filtered[
    df_couples_filtered['type of couple'].str.contains('same-sex', case=False)
].copy()

two_sex_marriages = df_couples[
    df_couples['type of couple'].str.contains('two sexes', case=False)
].copy()

# Create visualization for same-sex marriages
fig, ax = plt.subplots(figsize=(14, 7))

for couple_type in same_sex_marriages['type of couple'].unique():
    data = same_sex_marriages[same_sex_marriages['type of couple'] == couple_type]
    ax.plot(data['year'], data['Number'], 
            marker='o', linewidth=2.5, markersize=8, label=couple_type)

ax.set_title('Same-Sex Marriages in Sweden (2009-2024)\nSince the Gender-Neutral Marriage Law (May 1, 2009)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Marriages', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Add annotation for COVID-19
ax.axvline(x=2020, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.text(2020.5, ax.get_ylim()[1]*0.9, 'COVID19', 
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n🏳️‍🌈 Same-Sex Marriage Statistics:")
recent_year = same_sex_marriages['year'].max()
recent_data = same_sex_marriages[same_sex_marriages['year'] == recent_year]
total_same_sex = recent_data['Number'].sum()
print(f"\n📊 In {recent_year}:")
for _, row in recent_data.iterrows():
    print(f"  💑 {row['type of couple']}: {row['Number']:,.0f} marriages")
print(f"\n💕 Total same-sex marriages in {recent_year}: {total_same_sex:,.0f}")

In [ ]:
# Create visualization for couples with two sexes
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(two_sex_marriages['year'], two_sex_marriages['Number'], 
        marker='o', linewidth=2.5, markersize=8, color='#9370DB')

ax.set_title('Marriages in Couples with Two Sexes (2000-2024)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Marriages', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add annotation for COVID-19
ax.axvline(x=2020, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.text(2020.5, ax.get_ylim()[1]*0.9, 'COVID19', 
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n💑 Two-Sex Couple Marriage Statistics:")
recent_two_sex = two_sex_marriages[two_sex_marriages['year'] == recent_year]
print(f"\n📊 In {recent_year}:")
print(f"  💑 Marriages: {recent_two_sex['Number'].values[0]:,.0f}")

In [ ]:
# Fetch demographic data
print("📥 Fetching demographic marriage and divorce data (2000-2024)...\n")

df_new_age_groups = client.get_data_as_dataframe(
    table_id="TAB5829",
    value_codes={
        "Tid": ["*"],
        "Region": ["00"],
        "Kon": ["1", "2"],
        "Alder": ["*"],
        "TypPar": ["*"],
        "Civilstand": ["G", "SK"],
        "ContentsCode": ["0000053G"]
    }
)

print("\n✅ Successfully downloaded the data")

In [ ]:
# Analyze age patterns in marriages and divorces for 2024
recent_year = df_new_age_groups['year'].max()
df_age_recent = df_new_age_groups[df_new_age_groups['year'] == recent_year].copy()

# Separate marriages and divorces
df_marriages_age = df_age_recent[df_age_recent['marital status'] == 'married'].copy()
df_divorces_age = df_age_recent[df_age_recent['marital status'] == 'divorced'].copy()

# Group by age and sex for marriages
age_sex_marriages = df_marriages_age.groupby(['age', 'sex'])['Number'].sum().reset_index()
age_pivot_marriages = age_sex_marriages.pivot(index='age', columns='sex', values='Number')

# Group by age and sex for divorces
age_sex_divorces = df_divorces_age.groupby(['age', 'sex'])['Number'].sum().reset_index()
age_pivot_divorces = age_sex_divorces.pivot(index='age', columns='sex', values='Number')

# Create dual plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Plot marriages
age_pivot_marriages.plot(kind='bar', ax=ax1, width=0.8, color=['#4169E1', '#FF69B4'])
ax1.set_title(f'Marriages by Age Group and Sex in {recent_year}', 
             fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Age Group', fontsize=13, fontweight='bold')
ax1.set_ylabel('Number of Marriages', fontsize=13, fontweight='bold')
ax1.legend(['Men', 'Women'], fontsize=12)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=90)

# Plot divorces
age_pivot_divorces.plot(kind='bar', ax=ax2, width=0.8, color=['#4169E1', '#FF69B4'])
ax2.set_title(f'Divorces by Age Group and Sex in {recent_year}', 
             fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Age Group', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Divorces', fontsize=13, fontweight='bold')
ax2.legend(['Men', 'Women'], fontsize=12)
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

# Find most popular age groups
print(f"\n💍 Most Popular Age Groups for Marriage in {recent_year}:")
top_ages_marriage = age_sex_marriages.groupby('age')['Number'].sum().sort_values(ascending=False).head(5)
for i, (age, count) in enumerate(top_ages_marriage.items(), 1):
    print(f"  {i}. {age}: {count:,.0f} marriages")

print(f"\n💔 Most Common Age Groups for Divorce in {recent_year}:")
top_ages_divorce = age_sex_divorces.groupby('age')['Number'].sum().sort_values(ascending=False).head(5)
for i, (age, count) in enumerate(top_ages_divorce.items(), 1):
    print(f"  {i}. {age}: {count:,.0f} divorces")

---

# 💍 Part 3: The Long View - 150+ Years of Marriage Age Trends

Finally, let's take a journey through time! How has the average age at first marriage changed from 1871 to 2024? This is a fascinating long-term trend in Swedish demographic history! 📜✨

In [ ]:
# Fetch historical age at marriage data
print("📥 Fetching 150+ years of marriage age data (1871-2024)...\n")

df_age_marriage = client.get_data_as_dataframe(
    table_id="TAB4400",
    value_codes={
        "Tid": ["*"],
        "Kon": ["1", "2"],
        "ContentsCode": ["000000NQ"]
    },
    clean_colnames=True     # set this parameter to True to get clean column names
)

print("\n✅ Successfully downloaded the data")

In [ ]:
# Prepare the data
df_age = df_age_marriage.copy()

# Ensure numeric year
df_age['year'] = pd.to_numeric(df_age['year'])

# Create pivot for easier plotting
df_age_pivot = df_age.pivot(index='year', columns='sex', values='average_age_first_marriage')

# Create the magnificent historical plot
fig, ax = plt.subplots(figsize=(16, 9))

# Add shaded areas for World Wars FIRST (so they appear behind the lines)
ax.axvspan(1914, 1918, alpha=0.1, color='gray')
ax.axvspan(1939, 1945, alpha=0.1, color='gray')

# Plot both sexes
ax.plot(df_age_pivot.index, df_age_pivot['men'], 
        linewidth=3, label='Men', color='#4169E1', marker='o', 
        markersize=3, markevery=10)
ax.plot(df_age_pivot.index, df_age_pivot['women'], 
        linewidth=3, label='Women', color='#FF69B4', marker='s', 
        markersize=3, markevery=10)

# Add historical context annotations (without WWI and WWII now)
historical_events = [
    (1974, 'Cohabitation\nBecomes\nCommon', 0.8),
    (1989, 'Widow Pension\nReform\n(Data Spike)', 0.65),
    (2009, 'Marriage\nEquality\nlegalized', 0.65)
]

for year, event, height in historical_events:
    # Use different color for the 1989 data artifact
    line_color = 'red' if year == 1989 else 'gray'
    line_alpha = 0.7 if year == 1989 else 0.5
    box_color = 'lightcoral' if year == 1989 else 'wheat'
    
    ax.axvline(x=year, color=line_color, linestyle='--', linewidth=1, alpha=line_alpha)
    ax.text(year, ax.get_ylim()[1] * height, event, 
            fontsize=9, rotation=0, ha='center',
            bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.1))

# Add text labels for the World Wars
ax.text(1916, ax.get_ylim()[1] * 0.95, 'WWI', 
        fontsize=10, ha='center', fontweight='bold', color='black')
ax.text(1942, ax.get_ylim()[1] * 0.95, 'WWII', 
        fontsize=10, ha='center', fontweight='bold', color='black')

ax.set_title('Average Age at First Marriage in Sweden (1871-2024)\n"The Long Journey of Love Through Time"', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Age (years)', fontsize=14, fontweight='bold')
ax.legend(fontsize=13, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate key statistics
print("\n📈 The Evolution of Marriage Age:")
print("\n1871 (Beginning of records):")
data_1871 = df_age_pivot.loc[1871]
print(f"  👨 Men: {data_1871['men']:.1f} years")
print(f"  👩 Women: {data_1871['women']:.1f} years")
print(f"  📊 Age gap: {data_1871['men'] - data_1871['women']:.1f} years")

print("\n2024 (Latest data):")
data_2024 = df_age_pivot.loc[df_age_pivot.index.max()]
print(f"  👨 Men: {data_2024['men']:.1f} years")
print(f"  👩 Women: {data_2024['women']:.1f} years")
print(f"  📊 Age gap: {data_2024['men'] - data_2024['women']:.1f} years")

print("\n🔄 Changes over 150+ years:")
print(f"  👨 Men: +{data_2024['men'] - data_1871['men']:.1f} years")
print(f"  👩 Women: +{data_2024['women'] - data_1871['women']:.1f} years")
print(f"  💑 Gender gap narrowed by: {(data_1871['men'] - data_1871['women']) - (data_2024['men'] - data_2024['women']):.1f} years")

print("\n⚠️ The Curious Case of 1989:")
print("  The dramatic spike in 1989 wasn't a sudden change in romantic preferences!")
print("  Sweden abolished widow's pension (änkepension) effective January 1, 1990, but")
print("  women born in 1944 or earlier could still qualify under the old rules - IF they")
print("  were married. This led to a rush of marriages among older cohabiting couples in")
print("  1989, significantly raising the average marriage age for that year. A perfect")
print("  example of how policy changes can create unexpected demographic behaviors! 💡")

---

# 🎊 Conclusions: What We Learned About Love in Sweden

## Takeaways:

### 1. 📅 Seasonal Love Patterns
- Summer months (June-August) are consistently the most popular for weddings
- Winter months see fewer marriages but also fewer divorces
- Valentine's Day's February doesn't seem to create a wedding surge!

### 2. 🏳️‍🌈 The Diversity of Love
- Yearly same-sex marriages have been quite stable since legalization in 2009
- There are more female couple marriages than male couple marriages throughout the period

### 3. ⏰ The Changing Face of Marriage
- People are getting married significantly later in 2024 than in the 1800s
- From 1871 to 2024, the average marriage age increased by 8 years for men, and 7.6 years for women!

---

## 💝 Final Thoughts

Whether you're celebrating Valentine's Day with a partner, friends, family, or just yourself, this data tells a story about how love and commitment have evolved in Sweden over the past 150+ years. 

**And although the form of relationships may change, the human desire for connection remains!**

### Happy Valentine's Day, and do take care of yourselves! 💕

---

*Data source: Statistics Sweden (SCB) via the pxstatspy library*

*Created with ❤️ for Valentine's Day 2026*

---

# 🛠️ About pxstatspy

This analysis was made possible by **pxstatspy**, a comprehensive Python wrapper for the Statistics Sweden PxWebAPI 2.0.

### Key Features:
- 🔌 Easy connection to Swedish statistical databases
- 📊 Direct conversion to pandas DataFrames
- 🎯 Flexible data filtering and selection
- 🚀 Automatic handling of large datasets
- 🌍 Multi-language support

### Learn More:
- **GitHub:** https://github.com/xemarap/pxstatspy
- **Documentation:** See the README for comprehensive guides
- **Install:** `pip install pxstatspy`

### Perfect for:
- Data scientists and analysts
- Researchers studying Swedish demographics
- Students learning data analysis
- Anyone curious about Swedish statistics!

Happy analyzing! 📊💕